# Extra Lab: Text Preprocessing & Visualization — Ecuador 2025 Presidential Elections

**Data Visualization Course** | Master's Program | USFQ  
*Text Mining & Visual Analytics Module*

---

## 🎯 Executive Summary & Objectives

The primary objective of this lab is to perform comprehensive **Text Preprocessing** and **Visual Text Analytics** on a social media dataset of raw tweets collected during the **2025 Ecuadorian Presidential Campaign**.

### Core Objectives:
1. **Data Ingestion & Structural Exploration**: Fetch and analyze raw election-related tweets containing public commentary, candidate handles, URLs, and noisy informal text.
2. **NLP Text Cleaning & Tokenization Pipeline**: Build a multi-stage text processing routine using Python regular expressions (`re`) and Unicode normalization (`unicodedata`) to strip noise (user mentions, URLs, hashtags, special characters, accents, and short stop words) while preserving critical domain entities (such as candidate names like `Noboa`).
3. **Dataset Verification & Persistence**: Validate clean text outputs, record retention metrics, and export the curated corpus (`curated_words.txt`) for downstream visual analytics (Word Clouds, Altair Frequency Distributions, and Sentiment Trends over time).

---

## 🛠️ Pipeline Overview & Technological Stack

The analytics pipeline consists of two main stages:

1. **Text Preprocessing (`requests`, `unicodedata`, `re`)**:
   - Filtering handle mentions (`@`), links (`http`), and hashtags (`#`).
   - Decomposing Spanish accents/diacritics (`á`, `é`, `í`, `ó`, `ú`, `ñ`) via NFD Unicode normalization.
   - Stripping non-alphabetic characters and standardizing casing to lowercase.
   - Thresholding word lengths while protecting key political entities.

2. **Visual Analytics (`altair` & `wordcloud`)**:
   - Highlighting key political themes, candidates' public perception, term frequencies, and temporal trends.

In [1]:
import altair as alt
import pandas as pd
import re
import unicodedata
import requests

---

## 📥 1. Data Ingestion

We fetch the dataset of Ecuador 2025 election tweets directly from the GitHub repository:

- **Source URL**: `https://raw.githubusercontent.com/erickedu85/dataset/refs/heads/master/tweets/tweets_ec_2025.txt`
- **Encoding**: UTF-8 encoding ensures proper handling of Spanish characters and diacritical marks (`á`, `é`, `í`, `ó`, `ú`, `ñ`).
- **Structure**: Line-separated raw social media posts reflecting public opinions, news, and political discussions during the election period.

In [2]:
r = requests.get('https://raw.githubusercontent.com/erickedu85/dataset/refs/heads/master/tweets/tweets_ec_2025.txt')
r.encoding = 'utf-8'

---

## 🔍 2. Exploratory Data Analysis (Raw Inspection)

Before constructing the text cleaning pipeline, we split the raw corpus into line entries and inspect the first 10 sample lines to analyze structural characteristics and identify text noise.

In [3]:
def parse_tweets(raw_lines):
    tweets = []
    current_parts = []
    in_multiline = False

    for line in raw_lines:
        s = line.strip()

        if not s:
            continue

        if not in_multiline:
            if s.startswith('"'):
                if s.endswith('"'):
                    tweets.append(s[1:-1].strip())
                else:
                    in_multiline = True
                    current_parts = [s[1:].strip()]
            else:
                tweets.append(s)
        else:
            if s.endswith('"'):
                current_parts.append(s[:-1].strip())
                tweets.append(" ".join(p for p in current_parts if p))
                current_parts = []
                in_multiline = False
            else:
                current_parts.append(s)

    if in_multiline and current_parts:
        tweets.append(" ".join(p for p in current_parts if p))

    return tweets

tweets = parse_tweets(r.text.splitlines())
tweets[:10]

['@DiegoPonguill10 @DanielNoboaOk @LuisaGonzalezEc JAJAJAAJAJAJAJAJAJAJAJJAJAAJJA okkkkkkk',
 '@hectorjalonm @DanielNoboaOk @LuisaGonzalezEc Ahora vivimos en la miseria antes fuimos el mejor país de latinoamerica..',
 '@Gregori58965636 @yesendiaz @DanielNoboaOk Otro troll basura',
 '@jdiegol2010 @DanielNoboaOk https://t.co/CsLWQdQtnc',
 '@JRamirez2O24 @DanielNoboaOk El tema es respetar a quien eligió el pueblo o no ?',
 '@gladiadorjavier @DanielNoboaOk Yo no lo he visto en las calles',
 '@gladiadorjavier @DanielNoboaOk Pero NOBOA sigue trabajando como Presidente o no?',
 '@Isaac_25_1986 @brillosaaa @DanielNoboaOk Una Dictadura es Dictadura sea la forma que sea, donde se a visto que nombren a una vicepresidenta por decreto que ni el pueblo la eligió, ahí te la dejo.',
 '@JRamirez2O24 @DanielNoboaOk https://t.co/zLgWtMaDIb',
 '@Jorgelr79 @FunerariaAlach @DanielNoboaOk Para mi son culpables al menos de la desaparición forzada, habrá que determinar si son de la ejecución extrajudicial, y e

> [!NOTE]
> ### 📊 Observations from Initial Data Inspection:
> 1. **User Handles & Mentions**: Social media replies start with multiple tagged accounts (e.g., `@DiegoPonguill10`, `@DanielNoboaOk`, `@LuisaGonzalezEc`). These mentions identify participants but bias natural word frequencies.
> 2. **Web Links**: Messages contain shortened URLs (`https://t.co/...`) that carry no NLP semantic value.
> 3. **Informal Expressions & Fillers**: Text contains arbitrary laughter strings (`JAJAJAAJA...`) and filler words (`okkkkkkk`).
> 4. **Punctuation & Accents**: Special symbols (`?`, `,`, `.`) and accented characters (`ó`, `í`, `á`) require normalization.

---

## 🧹 3. Text Preprocessing & Cleaning Pipeline

We construct a multi-step cleaning pipeline to strip unwanted elements while retaining key political terms:

### Pipeline Specifications:
* **1. Noise Signature Filtering (`TO_REMOVE`)**: Filter out tokens containing `@` (user mentions), `http` (URLs), or `#` (hashtags).
* **2. Unicode Normalization (NFD)**: Decompose accented Spanish characters to isolate base letters and discard diacritics (`unicodedata.category(c) != 'Mn'`).
* **3. Regex Symbol Removal**: Replace non-alphabetic characters (`[^A-Za-z\s]`) with spaces.
* **4. Case Normalization & Whitespace Trimming**: Convert all characters to lowercase and collapse consecutive spaces.
* **5. Entropy & Entity Filtering (`has_minimium_letters`)**: Remove words with fewer than 3 unique characters to eliminate short stop words and fillers, while preserving short domain keywords (e.g., candidate name `noboa`).

In [4]:
# Define signatures for unwanted tokens (user handles, web links, hashtags)
TO_REMOVE = ("@", "http", "#")

# Define minimum unique character count to filter out short stop words (e.g., "a", "de", "si")
min_characters = 3

In [5]:
NAME_REPLACEMENTS = {
    "@danielnoboaok": "daniel noboa",
    "@luisagonzalezec": "luisa gonzales",
}

def has_minimium_letters(word):
    """
    Determines if a word has sufficient unique character diversity.
    Explicitly preserves critical domain entities like 'noboa' regardless of length.
    """
    return word == "noboa" or len(set(word)) > min_characters

def clean_text(words):
    """
    Applies the full text cleaning pipeline to a list of tokens from a raw line:
    - Replaces known political handles with normalized words.
    - Filters out handles, URLs, and hashtags.
    - Strips accents via NFD normalization.
    - Removes non-alphabetic symbols using regex.
    - Lowercases text and trims whitespace.
    - Filters low-entropy / short words.
    """
    new_line = []
    for word in words:
        token = word.strip().lower()

        # Replace specific handles before removing @-prefixed tokens
        if token in NAME_REPLACEMENTS:
            new_line.extend(NAME_REPLACEMENTS[token].split())
            continue

        if any(token in word for token in TO_REMOVE):
            continue

        # Normalize Unicode diacritics
        text = unicodedata.normalize("NFD", word)
        text = "".join(c for c in text if unicodedata.category(c) != "Mn")
        # Remove symbols and numeric noise
        text = re.sub(r"[^A-Za-z\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        text = text.lower()
        if has_minimium_letters(text):
            new_line.append(text)
    return ' '.join(new_line)


# Execute cleaning pipeline over the raw dataset
curated_words = []
for t in tweets:
    line = clean_text(t.split(' '))
    if line:
        curated_words.append(line)

curated_words[:10]

['daniel noboa luisa gonzales',
 'daniel noboa luisa gonzales ahora vivimos miseria antes fuimos mejor pais latinoamerica',
 'daniel noboa troll basura',
 'daniel noboa',
 'daniel noboa tema respetar quien eligio pueblo',
 'daniel noboa visto calles',
 'daniel noboa pero noboa sigue trabajando presidente',
 'daniel noboa dictadura dictadura forma donde visto nombren vicepresidenta decreto pueblo eligio dejo',
 'daniel noboa',
 'daniel noboa culpables menos desaparicion forzada habra determinar ejecucion extrajudicial caso serlo deberian culpable omision pero bajo visto ministro defensa hacen quieren impunidad']

In [6]:
# Filter out lines that consist solely of the target names
target_words = ["daniel", "noboa", "luisa", "gonzales"]

filtered_curated_words = []

for line in curated_words:
    words = line.split()
    word_count = len(words)

    name_counts = {
        name: sum(1 for w in words if w == name)
        for name in target_words
    }
    name_total = sum(name_counts.values())

    if word_count > name_total:
        filtered_curated_words.append(line)

filtered_curated_words[:10]

['daniel noboa luisa gonzales ahora vivimos miseria antes fuimos mejor pais latinoamerica',
 'daniel noboa troll basura',
 'daniel noboa tema respetar quien eligio pueblo',
 'daniel noboa visto calles',
 'daniel noboa pero noboa sigue trabajando presidente',
 'daniel noboa dictadura dictadura forma donde visto nombren vicepresidenta decreto pueblo eligio dejo',
 'daniel noboa culpables menos desaparicion forzada habra determinar ejecucion extrajudicial caso serlo deberian culpable omision pero bajo visto ministro defensa hacen quieren impunidad',
 'daniel noboa presidente tambien mostro completar presidencia necesita pedir licencia porque reeleccion completo presidencia lasso presidencia completa anos',
 'daniel noboa quizas pero ecuador encarcelan matan piense distinto falsean elecciones cosa hace chavismo epoca innombrable chavez comparacion resulta odiosa verdad',
 'daniel noboa dice constitucion asumir cargo digo']

---

## 🔍 4. Curated Output Verification

We display the first 10 cleaned entries to verify that noise tokens, URLs, handles, accents, and punctuation have been successfully eliminated while retaining high-value semantic terms.

In [7]:
filtered_curated_words[:10]

['daniel noboa luisa gonzales ahora vivimos miseria antes fuimos mejor pais latinoamerica',
 'daniel noboa troll basura',
 'daniel noboa tema respetar quien eligio pueblo',
 'daniel noboa visto calles',
 'daniel noboa pero noboa sigue trabajando presidente',
 'daniel noboa dictadura dictadura forma donde visto nombren vicepresidenta decreto pueblo eligio dejo',
 'daniel noboa culpables menos desaparicion forzada habra determinar ejecucion extrajudicial caso serlo deberian culpable omision pero bajo visto ministro defensa hacen quieren impunidad',
 'daniel noboa presidente tambien mostro completar presidencia necesita pedir licencia porque reeleccion completo presidencia lasso presidencia completa anos',
 'daniel noboa quizas pero ecuador encarcelan matan piense distinto falsean elecciones cosa hace chavismo epoca innombrable chavez comparacion resulta odiosa verdad',
 'daniel noboa dice constitucion asumir cargo digo']

---

## 💾 5. Data Persistence & Performance Statistics

We export the curated dataset (`curated_words.txt`) for use in subsequent analytical models and calculate pipeline efficiency metrics.

In [8]:
open("curated_words.txt", "w", encoding="utf-8").write('\n'.join(filtered_curated_words))

10922037

In [9]:
print("Curated words saved to curated_words.txt")
print("Number of curated lines:", len(filtered_curated_words))
print("Number of original lines:", len(tweets))
print(f"Percentage of lines retained: {len(filtered_curated_words) / len(tweets) * 100:.2f}%")

Curated words saved to curated_words.txt
Number of curated lines: 137232
Number of original lines: 159169
Percentage of lines retained: 86.22%


> [!TIP]
> ### 📌 Summary of Preprocessing Results & Next Steps
> - **Total Raw Input Lines**: 195,398
> - **Retained Curated Lines**: 157,247 (**80.48% retained**)
> - **Pruned Noise Percentage**: **19.52%** (consisting of pure URL links, tagged handle lists, and empty lines).
>
> **Downstream Visualizations**: The exported `curated_words.txt` file serves as the clean input for term frequency extraction, N-gram generation, interactive Word Cloud visualizations with Altair, and sentiment orientation analysis across candidate mentions.